# Kaggle Submission — XGBoost (best model, lowest holdout WMAE)

Generates a Kaggle-format submission using the already-trained XGBoost
pipeline (`models/xgboost_pipeline.joblib`) from
`model_experiment_XGBoost.ipynb` -- no retraining needed here.

**Why XGBoost**: of every model tried in this project, it has the lowest
local-test holdout WMAE:

| Model | Holdout WMAE |
|---|---|
| **XGBoost** | **1639.12** |
| LightGBM | 1672.26 |
| N-BEATS (best generic) | 2161.25 |
| PatchTST | 2190.61 |
| DLinear | 2532.49 |
| TimesFM (direct) | 2618.40 |
| TimesFM (recursive) | 2787.23 |
| ARIMA | 2579.07 (different eval window, not directly comparable) |
| TFT (reduced-scope, no tuning) | 3996.37 |
| Prophet | 6932.91 |

**v2 -- fixes a real bug from the first version of this notebook.** The
first submission scored public=8259.84, private=8494.56 on Kaggle --
roughly 5x worse than the internal holdout WMAE above. Root cause: it
called `pipeline.predict(test)` on the whole 39-week test set in one
shot, which featurizes `test.csv` with `Weekly_Sales` entirely NaN
(genuinely unknown, as real test data is). `roll_mean_4`/`roll_mean_8`
need the past 4-8 weeks of real sales, so they go NaN from the **2nd**
test week onward; `lag13` survives until week 13, then breaks too -- 4 of
5 lag/rolling features NaN for most of the test period. The internal
holdout evaluation never caught this because `local_test_raw` (carved
from `train.csv`) has real `Weekly_Sales`, so its lag/rolling features
were always computed from ground truth -- a fundamentally easier task
that never exercises the all-unknown-future scenario Kaggle's real
`test.csv` actually is.

**Fix**: `utils.feature_engineering.recursive_predict()` -- predicts one
calendar week at a time, feeding each week's own predictions back into a
running history before featurizing the next week, so short lag/rolling
windows get real numbers (predictions standing in for the still-unknown
truth) instead of NaN. Validated directly: a genuinely-blind local
re-evaluation (`local_test_raw` treated as unseen, scored against its
real values afterward) gives WMAE **1580.80** -- in line with the
original 1639.12, confirming the ~5x gap was entirely this bug, not a
real generalization problem. Also fixed in the same pass: `merge_raw()`
was leaving `CPI`/`Unemployment` as NaN for the real `features.csv`'s
last ~13 weeks (2013-05-03 onward, a genuine gap in the competition data
itself, never exercised by `train.csv` which ends 2012-10-26) -- now
forward-filled per store before merging.

<a id='1'></a>
## 1. Load pipeline and data

In [1]:
import joblib
import numpy as np
import pandas as pd

from utils.feature_engineering import recursive_predict

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'

pipeline = joblib.load('models/xgboost_pipeline.joblib')
feature_selector = pipeline.named_steps['feature_selection']
model = pipeline.named_steps['model']

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])
features = pd.read_csv(DATA_DIR + 'features.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
sample_submission = pd.read_csv(DATA_DIR + 'sampleSubmission.csv')

print(f'train.csv: {train.shape}, {train.Date.min().date()} -> {train.Date.max().date()}')
print(f'test.csv : {test.shape}, {test.Date.min().date()} -> {test.Date.max().date()}, {test.Date.nunique()} weeks')

train.csv: (421570, 5), 2010-02-05 -> 2012-10-26
test.csv : (115064, 4), 2012-11-02 -> 2013-07-26, 39 weeks


<a id='2'></a>
## 2. Recursive predict

`train.csv` is the real, complete history (unlike `local_train_raw`,
which is only 91 weeks of it) -- used here as `initial_history_df` for
the very first test week's lag/rolling context, exactly the role
`FeatureEngineeringTransformer.fit()` already gave it when the pipeline
was originally trained.

In [2]:
preds_df = recursive_predict(
    test, train, features, stores, feature_selector, model, verbose=True,
)

print(f'\n{len(preds_df)} predictions generated (expected {len(test)})')
print(f'any NaN: {preds_df["Weekly_Sales"].isna().any()}')
print(f'min={preds_df["Weekly_Sales"].min():.2f}, max={preds_df["Weekly_Sales"].max():.2f}, '
      f'mean={preds_df["Weekly_Sales"].mean():.2f}')

  week 1/39 (2012-11-02T00:00:00.000000000): 2959 rows, nan_rate=0.0083, pred mean=16571.1, max=184099.7


  week 2/39 (2012-11-09T00:00:00.000000000): 2971 rows, nan_rate=0.0090, pred mean=16466.8, max=187577.9


  week 3/39 (2012-11-16T00:00:00.000000000): 2956 rows, nan_rate=0.0076, pred mean=16280.3, max=187335.5


  week 4/39 (2012-11-23T00:00:00.000000000): 2976 rows, nan_rate=0.0089, pred mean=22772.8, max=631000.3


  week 5/39 (2012-11-30T00:00:00.000000000): 2962 rows, nan_rate=0.0082, pred mean=17426.7, max=180213.3


...

  week 35/39 (2013-06-28T00:00:00.000000000): 2909 rows, nan_rate=0.0076, pred mean=16998.4, max=171711.9


  week 36/39 (2013-07-05T00:00:00.000000000): 2935 rows, nan_rate=0.0098, pred mean=17575.8, max=172604.4


  week 37/39 (2013-07-12T00:00:00.000000000): 2922 rows, nan_rate=0.0088, pred mean=16728.2, max=171713.5


  week 38/39 (2013-07-19T00:00:00.000000000): 2939 rows, nan_rate=0.0103, pred mean=16496.0, max=171508.6


  week 39/39 (2013-07-26T00:00:00.000000000): 2936 rows, nan_rate=0.0104, pred mean=16193.6, max=171077.6

115064 predictions generated (expected 115064)
any NaN: False
min=0.00, max=631000.31, mean=17104.08


<a id='3'></a>
## 3. Format submission

Kaggle's required format: `Id = f'{Store}_{Dept}_{Date}'`,
`Weekly_Sales = <predicted value>` -- verified directly against
`sampleSubmission.csv`'s own `Id` format below before writing anything.

In [3]:
submission = pd.DataFrame({
    'Id': preds_df['Store'].astype(str) + '_' + preds_df['Dept'].astype(str) + '_' + preds_df['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': preds_df['Weekly_Sales'],
})

assert len(submission) == len(sample_submission), \
    f'row count mismatch: {len(submission)} vs {len(sample_submission)}'
assert set(submission['Id']) == set(sample_submission['Id']), \
    'Id values do not match sampleSubmission.csv exactly'
assert submission['Weekly_Sales'].isna().sum() == 0, 'NaN predictions present'
assert (submission['Weekly_Sales'] >= 0).all(), 'negative predictions present'

print('All sanity checks passed: row count, Id format, no NaN, no negatives.')
submission.head()

All sanity checks passed: row count, Id format, no NaN, no negatives.


,Id,Weekly_Sales
0,1_1_2012-11-02,36453.011719
1,1_2_2012-11-02,46199.296875
2,1_3_2012-11-02,10139.344727
3,1_4_2012-11-02,39050.789062
4,1_5_2012-11-02,29584.693359


<a id='4'></a>
## 4. Save

In [4]:
submission.to_csv('submission.csv', index=False)
print('Saved to submission.csv -- ready to upload to Kaggle.')
print(f'{len(submission)} rows')

Saved to submission.csv -- ready to upload to Kaggle.
115064 rows
